<a href="https://colab.research.google.com/github/sajjkavinda/ML-based-Intrution-detection-system/blob/main/External_Dataset_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##External Dataset Preprocessing

In this notebook, using an external dataset CIC_IDS2017 we are performing testing and validation. This workthough will give us a result on how our trained model will handle unknown traffic apart from the dataset it was used to train.

This dataset (CIC_IDS2017) is the older version of the dataset we have used for training the model.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Import libraries

In [10]:
import pandas as pd
import numpy as np
import zipfile

from pathlib import Path

Set paths for the Dataset and Outputs

In [11]:
PROJECT_DIR = Path("/content/drive/MyDrive/MSc_Dissertation")

DATASET_DIR = PROJECT_DIR / "Dataset/CIC_IDS2017"

CSV_DIR = DATASET_DIR / "MachineLearningCSV/MachineLearningCVE"

PROCESSED_DIR = PROJECT_DIR / "Dataset/Processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(CSV_DIR)

/content/drive/MyDrive/MSc_Dissertation/Dataset/CIC_IDS2017/MachineLearningCSV/MachineLearningCVE


Load the dataset

In [12]:
csv_files = list(CSV_DIR.glob("*.csv"))

print("Files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Files found: 8
Wednesday-workingHours.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


Combine the dataset

In [13]:
df_list = []

for file in csv_files:

    print("Loading:", file.name)

    temp = pd.read_csv(
        file,
        encoding="latin1",
        low_memory=False
    )

    df_list.append(temp)


df = pd.concat(
    df_list,
    ignore_index=True
)


print("Dataset shape:")
print(df.shape)

Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Dataset shape:
(2830743, 79)


Clean column names

In [14]:
df.columns = df.columns.str.strip()

print(df.columns[:10])

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std'],
      dtype='object')


Check the labels

In [15]:
print(df["Label"].value_counts())

Label
BENIGN                          2273097
DoS Hulk                         231073
PortScan                         158930
DDoS                             128027
DoS GoldenEye                     10293
FTP-Patator                        7938
SSH-Patator                        5897
DoS slowloris                      5796
DoS Slowhttptest                   5499
Bot                                1966
Web Attack ï¿½ Brute Force         1507
Web Attack ï¿½ XSS                  652
Infiltration                         36
Web Attack ï¿½ Sql Injection         21
Heartbleed                           11
Name: count, dtype: int64


Remove duplicates

In [16]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Duplicates removed:", before-after)
print("Remaining samples:", after)

Duplicates removed: 308381
Remaining samples: 2522362


Remove infinite values

In [17]:
df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Infinity values replaced")

Infinity values replaced


Remove missing values

In [18]:
before = len(df)

df = df.dropna()

after = len(df)

print("Rows removed:", before-after)
print("Remaining samples:", after)

Rows removed: 1564
Remaining samples: 2520798


Convert labels to binary
BENIGN → normal traffic
Everything else → attack

In [19]:
df["Label"] = df["Label"].apply(
    lambda x: 0 if x.strip() == "BENIGN" else 1
)


print(df["Label"].value_counts())

Label
0    2095057
1     425741
Name: count, dtype: int64


Save the cleaned dataset

In [20]:
df.to_csv(
    PROCESSED_DIR / "CIC_IDS2017_cleaned.csv",
    index=False
)

print("Cleaned dataset saved")

Cleaned dataset saved


Feature alignment check with the original dataset

In [24]:
import pickle

with open(PROCESSED_DIR / "X_train.pkl", "rb") as f:
    X_train = pickle.load(f)


training_features = list(X_train.columns)


print("Training features:")
print(len(training_features))
print(training_features)

missing_features = set(training_features) - set(df.columns)

print("Missing features:")
print(missing_features)


Training features:
30
['Fwd Seg Size Min', 'Init Fwd Win Byts', 'Tot Bwd Pkts', 'Bwd Pkts/s', 'Pkt Len Mean', 'Flow IAT Mean', 'Fwd Pkt Len Max', 'Fwd IAT Mean', 'Pkt Len Max', 'Fwd Pkt Len Mean', 'Flow IAT Max', 'Flow Pkts/s', 'ACK Flag Cnt', 'Pkt Len Var', 'Tot Fwd Pkts', 'Bwd Pkt Len Max', 'Flow Byts/s', 'Flow Duration', 'Bwd IAT Min', 'Bwd Pkt Len Mean', 'Init Bwd Win Byts', 'Bwd IAT Mean', 'URG Flag Cnt', 'Fwd IAT Std', 'Down/Up Ratio', 'Bwd IAT Std', 'Bwd IAT Tot', 'Idle Mean', 'Bwd IAT Max', 'Flow IAT Std']
Missing features:
{'Init Fwd Win Byts', 'Tot Bwd Pkts', 'Flow Pkts/s', 'Fwd Seg Size Min', 'Flow Byts/s', 'Pkt Len Var', 'Bwd Pkt Len Max', 'Init Bwd Win Byts', 'Bwd Pkts/s', 'Bwd IAT Tot', 'Pkt Len Mean', 'Bwd Pkt Len Mean', 'Fwd Pkt Len Mean', 'ACK Flag Cnt', 'Pkt Len Max', 'URG Flag Cnt', 'Fwd Pkt Len Max', 'Tot Fwd Pkts'}


Missing features found. So need to change the column names in order to work with our trained model

In [23]:
print("Number of columns:", len(df.columns))

for col in df.columns:
    print(col)

Number of columns: 79
Destination Port
Flow Duration
Total Fwd Packets
Total Backward Packets
Total Length of Fwd Packets
Total Length of Bwd Packets
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Bytes/s
Flow Packets/s
Flow IAT Mean
Flow IAT Std
Flow IAT Max
Flow IAT Min
Fwd IAT Total
Fwd IAT Mean
Fwd IAT Std
Fwd IAT Max
Fwd IAT Min
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Bwd IAT Min
Fwd PSH Flags
Bwd PSH Flags
Fwd URG Flags
Bwd URG Flags
Fwd Header Length
Bwd Header Length
Fwd Packets/s
Bwd Packets/s
Min Packet Length
Max Packet Length
Packet Length Mean
Packet Length Std
Packet Length Variance
FIN Flag Count
SYN Flag Count
RST Flag Count
PSH Flag Count
ACK Flag Count
URG Flag Count
CWE Flag Count
ECE Flag Count
Down/Up Ratio
Average Packet Size
Avg Fwd Segment Size
Avg Bwd Segment Size
Fwd Header Length.1
Fwd Avg Bytes/Bulk
Fwd Avg Packet

Feature mapping

In [25]:
column_mapping = {

    "Total Fwd Packets": "Tot Fwd Pkts",
    "Total Backward Packets": "Tot Bwd Pkts",

    "Fwd Packet Length Max": "Fwd Pkt Len Max",
    "Fwd Packet Length Mean": "Fwd Pkt Len Mean",

    "Bwd Packet Length Max": "Bwd Pkt Len Max",
    "Bwd Packet Length Mean": "Bwd Pkt Len Mean",

    "Flow Bytes/s": "Flow Byts/s",
    "Flow Packets/s": "Flow Pkts/s",

    "Packet Length Mean": "Pkt Len Mean",
    "Packet Length Variance": "Pkt Len Var",
    "Max Packet Length": "Pkt Len Max",

    "Average Packet Size": "Pkt Size Avg",

    "Avg Fwd Segment Size": "Fwd Seg Size Min",

    "Avg Bwd Segment Size": "Bwd Pkt Len Mean",

    "Init_Win_bytes_forward": "Init Fwd Win Byts",
    "Init_Win_bytes_backward": "Init Bwd Win Byts",

    "Bwd IAT Total": "Bwd IAT Tot",

    "ACK Flag Count": "ACK Flag Cnt",
    "URG Flag Count": "URG Flag Cnt",

    "Fwd IAT Total": "Fwd IAT Tot",

    "min_seg_size_forward": "Fwd Seg Size Min",

    "Bwd Packets/s": "Bwd Pkts/s",
    "Fwd Packets/s": "Fwd Pkts/s",

}

df.columns = df.columns.str.strip()

df = df.rename(columns=column_mapping)

print("Columns renamed")

Columns renamed


Validation

In [26]:
missing_features = set(training_features) - set(df.columns)

print("Missing features:")
print(missing_features)

Missing features:
set()


External validation features and labels

In [27]:
X_external = df[training_features]

y_external = df["Label"]


print("X shape:", X_external.shape)
print("y shape:", y_external.shape)

X shape: (2520798, 32)
y shape: (2520798,)


Save the validation dataset

In [28]:
X_external.to_csv(
    PROCESSED_DIR / "CIC_IDS2017_external_X.csv",
    index=False
)


y_external.to_csv(
    PROCESSED_DIR / "CIC_IDS2017_external_y.csv",
    index=False
)


print("CIC-IDS2017 external validation dataset saved")

CIC-IDS2017 external validation dataset saved
